In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, date
from pydantic import BaseModel
from typing import Dict, List, Optional

## Explicação

In [2]:
class User(BaseModel):
    id: int
    name: str 
    age: int
    signup_ts: Optional[datetime]

In [3]:
external_data = {
    'id': 123,
    'name': "Carolina", #E se mudarmos para um int?
    'age': 56, #E se mudarmos para uma str?
    'signup_ts': '2019-06-01 12:22' # E se mudarmos para data ?
}

User(**external_data)

User(id=123, name='Carolina', age=56, signup_ts=datetime.datetime(2019, 6, 1, 12, 22))

## Como aplicar

### Preparando a Base

In [4]:
df = pd.read_csv("../../0. Dados/telco_dataset.csv")
df.drop(['customerID'],axis=1, inplace=True)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
lista_X_quanti = ["tenure",
                  "MonthlyCharges",
                  "TotalCharges"]
lista_X_quali = ['OnlineSecurity',
                 'TechSupport']

In [5]:
df['TotalCharges'] = df['TotalCharges'].replace(" ", 0)

In [7]:
for col in lista_X_quali:
    try:
        df[col] = df[col].astype("str")
        df[col] = df[col].str.strip()
    except:
        print(col)

for col in lista_X_quanti:
    try:df[col] = df[col].astype("float")
    except:print(col)

In [8]:
y = df['Churn'].replace(to_replace = ['Yes','No'],value = [1,0])
X = df[lista_X_quanti + lista_X_quali]

C:\Users\Carolina\AppData\Local\Temp\ipykernel_18508\4212662862.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df['Churn'].replace(to_replace = ['Yes','No'],value = [1,0])


### Montando os schemas

In [13]:
class RawDataSchema(BaseModel):
    """
    Data Model Input schema
    """
    tenure: int
    MonthlyCharges : float
    TotalCharges : float
    OnlineSecurity: str
    TechSupport : str

In [15]:
class MultipleDataSchema(BaseModel):
    inputs: List[RawDataSchema]

In [16]:
MultipleDataSchema(
    inputs=X.replace({np.nan: None}).to_dict(orient="records")
)

MultipleDataSchema(inputs=[RawDataSchema(tenure=1, MonthlyCharges=29.85, TotalCharges=29.85, OnlineSecurity='No', TechSupport='No'), RawDataSchema(tenure=34, MonthlyCharges=56.95, TotalCharges=1889.5, OnlineSecurity='Yes', TechSupport='No'), RawDataSchema(tenure=2, MonthlyCharges=53.85, TotalCharges=108.15, OnlineSecurity='Yes', TechSupport='No'), RawDataSchema(tenure=45, MonthlyCharges=42.3, TotalCharges=1840.75, OnlineSecurity='Yes', TechSupport='Yes'), RawDataSchema(tenure=2, MonthlyCharges=70.7, TotalCharges=151.65, OnlineSecurity='No', TechSupport='No'), RawDataSchema(tenure=8, MonthlyCharges=99.65, TotalCharges=820.5, OnlineSecurity='No', TechSupport='No'), RawDataSchema(tenure=22, MonthlyCharges=89.1, TotalCharges=1949.4, OnlineSecurity='No', TechSupport='No'), RawDataSchema(tenure=10, MonthlyCharges=29.75, TotalCharges=301.9, OnlineSecurity='Yes', TechSupport='No'), RawDataSchema(tenure=28, MonthlyCharges=104.8, TotalCharges=3046.05, OnlineSecurity='No', TechSupport='Yes'), Raw